#Importação das bibliotecas

In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

#Primeira EDA

In [2]:
df = pd.read_csv('/content/ExpVinho.csv', sep='\t')

In [3]:
df.head(5)

,Id,País,1970,1970.1,1971,1971.1,1972,1972.1,1973,1973.1,...,2021,2021.1,2022,2022.1,2023,2023.1,2024,2024.1,2025,2025.1
0,1,Afeganistão,0,0,0,0,0,0,0,0,...,11,46,0,0,0,0,0,0,0,0
1,2,África do Sul,0,0,0,0,0,0,0,0,...,0,0,0,0,117,698,103,1783,0,0
2,3,"Alemanha, República Democrática",0,0,0,0,4168,2630,12000,8250,...,2698,6741,7630,45367,4806,31853,6666,48095,3983,24672
3,4,Angola,0,0,0,0,0,0,0,0,...,0,0,4068,4761,0,0,0,0,0,0
4,5,Anguilla,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [4]:
df.shape

(144, 114)

In [5]:
df.dtypes

,0
Id,int64
País,object
1970,int64
1970.1,int64
1971,int64
...,...
2023.1,int64
2024,int64
2024.1,int64
2025,int64


In [6]:
df.isnull().sum()

,0
Id,0
País,0
1970,0
1970.1,0
1971,0
...,...
2023.1,0
2024,0
2024.1,0
2025,0


In [7]:
df.duplicated().sum()

np.int64(0)

In [8]:
df.columns.tolist()[:10]

['Id',
 'País',
 '1970',
 '1970.1',
 '1971',
 '1971.1',
 '1972',
 '1972.1',
 '1973',
 '1973.1']

#Reshape da tabela

##Filtrando as colunas de litros e de valor e adicionando em listas.

In [9]:
cols_litros = []

for coluna in df.columns:
  if "." not in coluna and coluna != 'Id' and coluna != 'País':
    cols_litros.append(coluna)

print(cols_litros)

['1970', '1971', '1972', '1973', '1974', '1975', '1976', '1977', '1978', '1979', '1980', '1981', '1982', '1983', '1984', '1985', '1986', '1987', '1988', '1989', '1990', '1991', '1992', '1993', '1994', '1995', '1996', '1997', '1998', '1999', '2000', '2001', '2002', '2003', '2004', '2005', '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025']


In [10]:
cols_valor = []

for coluna in df.columns:
  if "." in coluna and coluna != 'Id' and coluna != 'País':
    cols_valor.append(coluna)

print(cols_valor)

['1970.1', '1971.1', '1972.1', '1973.1', '1974.1', '1975.1', '1976.1', '1977.1', '1978.1', '1979.1', '1980.1', '1981.1', '1982.1', '1983.1', '1984.1', '1985.1', '1986.1', '1987.1', '1988.1', '1989.1', '1990.1', '1991.1', '1992.1', '1993.1', '1994.1', '1995.1', '1996.1', '1997.1', '1998.1', '1999.1', '2000.1', '2001.1', '2002.1', '2003.1', '2004.1', '2005.1', '2006.1', '2007.1', '2008.1', '2009.1', '2010.1', '2011.1', '2012.1', '2013.1', '2014.1', '2015.1', '2016.1', '2017.1', '2018.1', '2019.1', '2020.1', '2021.1', '2022.1', '2023.1', '2024.1', '2025.1']


##Utilizando melt para agrupar a tabela em formato long.

In [11]:
df_litros = df.melt(id_vars="País", value_vars=cols_litros, var_name="ano", value_name="litros")

In [12]:
df_valor = df.melt(id_vars="País", value_vars=cols_valor, var_name="ano", value_name="valor_usd")

In [13]:
print(df_litros.shape)
print(df_valor.shape)

(8064, 3)
(8064, 3)


##Utilizando merge para unir as duas tabelas.

In [14]:
df_valor['ano'] = df_valor['ano'].str.replace('.1', '', regex=False)

In [15]:
df_long = df_litros.merge(df_valor, on=["País", "ano"])

In [16]:
df_long['ano'] = df_long['ano'].astype(int)

###Adicionando coluna País de Origem e filtrando os últimos 15 anos.

In [17]:
df_long.insert(0, 'pais_origem', 'Brasil')

In [18]:
df_long.head()

,pais_origem,País,ano,litros,valor_usd
0,Brasil,Afeganistão,1970,0,0
1,Brasil,África do Sul,1970,0,0
2,Brasil,"Alemanha, República Democrática",1970,0,0
3,Brasil,Angola,1970,0,0
4,Brasil,Anguilla,1970,0,0


In [19]:
df_long = df_long[df_long['ano'] >= df_long['ano'].max() - 14]

In [20]:
print(df_long['ano'].unique())

[2011 2012 2013 2014 2015 2016 2017 2018 2019 2020 2021 2022 2023 2024
 2025]


In [21]:
print(df_long.shape)

(2160, 5)


###Agrupando linhas com groupby para verificar valor por período.

In [22]:
por_pais = df_long.groupby('País')['valor_usd'].sum().sort_values(ascending=False)

In [23]:
por_pais.head(10)

,valor_usd
País,
Paraguai,53615443
Rússia,17421596
Estados Unidos,9624239
Haiti,5174689
China,4741491
Reino Unido,4651796
Espanha,3814065
Países Baixos,2598901
Uruguai,2010364


In [24]:
por_pais_litros = df_long.groupby('País')['litros'].sum().sort_values(ascending=False)

In [25]:
por_pais_litros.head(10)

,litros
País,
Paraguai,38060943
Rússia,10909662
Haiti,3574327
Estados Unidos,3419036
Espanha,1989252
Uruguai,1325881
China,1074503
Reino Unido,1068084
Japão,684699


###Verificando valor por litro, visto que alguns países não seguem a mesma posição no ranking


In [26]:
preco_medio = (por_pais / por_pais_litros).sort_values(ascending=False)

In [27]:
preco_medio = preco_medio.rename('preco_medio_usd')

In [28]:
preco_medio.head(10)

,preco_medio_usd
País,
"Cook, Ilhas",18.000000
Argélia,14.500000
Bulgária,12.814286
Belice,12.300000
Bangladesh,11.083333
África do Sul,10.388000
Geórgia,9.875421
Emirados Árabes Unidos,9.500557
Mauritânia,9.444444


In [29]:
resumo_pais = pd.concat([por_pais, por_pais_litros, preco_medio], axis=1)
resumo_pais.head(10)

,valor_usd,litros,preco_medio_usd
País,,,
Paraguai,53615443,38060943,1.408674
Rússia,17421596,10909662,1.596896
Estados Unidos,9624239,3419036,2.814898
Haiti,5174689,3574327,1.447738
China,4741491,1074503,4.412729
Reino Unido,4651796,1068084,4.355272
Espanha,3814065,1989252,1.917336
Países Baixos,2598901,642447,4.045316
Uruguai,2010364,1325881,1.516248


#Criando função em Python para o EDA dos dados extras.

In [30]:
def preparar_dados(caminho, produto):
    df = pd.read_csv(caminho, sep='\t')

    cols_litros = []
    for coluna in df.columns:
        if '.' not in coluna and coluna != 'Id' and coluna != 'País':
            cols_litros.append(coluna)

    cols_valor = []
    for coluna in df.columns:
        if '.' in coluna:
            cols_valor.append(coluna)

    df_litros = df.melt(id_vars='País', value_vars=cols_litros, var_name='ano', value_name='litros')

    df_valor = df.melt(id_vars='País', value_vars=cols_valor, var_name='ano', value_name='valor_usd')
    df_valor['ano'] = df_valor['ano'].str.replace('.1', '', regex=False)

    df_long = df_litros.merge(df_valor, on=['País', 'ano'])
    df_long['ano'] = df_long['ano'].astype(int)

    df_long.insert(0, 'pais_origem', 'Brasil')
    df_long.insert(1, 'produto', produto)

    df_long = df_long[df_long['ano'] >= df_long['ano'].max() - 14]

    return df_long

In [31]:
df_vinho = preparar_dados('/content/ExpVinho.csv', 'Vinho de Mesa')
df_espumantes = preparar_dados('/content/ExpEspumantes.csv', 'Espumante')
df_suco = preparar_dados('/content/ExpSuco.csv', 'Suco de Uva')
df_uvas = preparar_dados('/content/ExpUva.csv', 'Uvas Frescas')

In [32]:
df_total = pd.concat([df_vinho, df_espumantes, df_suco, df_uvas], axis=0)

In [33]:
df_total.head(10)

,pais_origem,produto,País,ano,litros,valor_usd
5904,Brasil,Vinho de Mesa,Afeganistão,2011,0,0
5905,Brasil,Vinho de Mesa,África do Sul,2011,0,0
5906,Brasil,Vinho de Mesa,"Alemanha, República Democrática",2011,36070,144150
5907,Brasil,Vinho de Mesa,Angola,2011,13889,69001
5908,Brasil,Vinho de Mesa,Anguilla,2011,0,0
5909,Brasil,Vinho de Mesa,Antígua e Barbuda,2011,0,0
5910,Brasil,Vinho de Mesa,Antilhas Holandesas,2011,7335,10188
5911,Brasil,Vinho de Mesa,Arábia Saudita,2011,0,0
5912,Brasil,Vinho de Mesa,Argélia,2011,0,0
5913,Brasil,Vinho de Mesa,Argentina,2011,13253,55460


In [34]:
df_total.groupby(['produto', 'País'])['valor_usd'].sum()

produto        País               
Espumante      Alemanha               316821
               Angola                 745652
               Antigua e Barbuda         832
               Antilhas Holandesas     65766
               Argentina              298318
                                       ...  
Vinho de Mesa  Venezuela              594334
               Vietnã                   1137
               África do Sul            2597
               Áustria                  5459
               Índia                    1615
Name: valor_usd, Length: 529, dtype: int64

In [35]:
por_produto_pais = df_total.groupby(['produto', 'País'])['valor_usd'].sum()
top10 = por_produto_pais.groupby('produto').nlargest(10).reset_index(level=0, drop=True).reset_index()
top10.head(20)

,produto,País,valor_usd
0,Espumante,Estados Unidos,10162452
1,Espumante,Paraguai,2113669
2,Espumante,Reino Unido,1658937
3,Espumante,Uruguai,961332
4,Espumante,Letônia,842251
5,Espumante,Angola,745652
6,Espumante,China,614258
7,Espumante,Bélgica,580019
8,Espumante,Japão,546918
9,Espumante,Chile,532040


#Usando Plotly para criar um gráfico de barras horizontais.

In [36]:
import plotly.express as px

fig = px.bar(
    top10,
    x='valor_usd',
    y='País',
    color='produto',
    facet_col='produto',
    facet_col_wrap=2,
    orientation='h',
    title='Top 10 países por valor exportado (US$) por produto'
)

fig.update_yaxes(matches=None, showticklabels=True)
fig.update_xaxes(matches=None, showticklabels=True)
fig.show()

In [37]:
evolucao_temporal = df_total.groupby(['ano', 'produto'])['valor_usd'].sum().reset_index()

In [38]:
evolucao_temporal.head(10)

,ano,produto,valor_usd
0,2011,Espumante,568390
1,2011,Suco de Uva,15737683
2,2011,Uvas Frescas,135782857
3,2011,Vinho de Mesa,3615120
4,2012,Espumante,813909
5,2012,Suco de Uva,7719833
6,2012,Uvas Frescas,121898272
7,2012,Vinho de Mesa,5521293
8,2013,Espumante,928626
9,2013,Suco de Uva,12427609


In [39]:
fig = px.line(
    evolucao_temporal,
    x='ano',
    y='valor_usd',
    color='produto',
    title='Evolução do valor exportado por produto (2011-2025)'
)

fig.show()

In [40]:
preco_por_litro_produtos = df_total.groupby(['País', 'produto'])[['valor_usd', 'litros']].sum().reset_index()

In [41]:
preco_por_litro_produtos['preco_medio'] = preco_por_litro_produtos['valor_usd'] / preco_por_litro_produtos['litros']
preco_por_litro_produtos = preco_por_litro_produtos[preco_por_litro_produtos['litros'] > 0]

In [42]:
preco_por_litro_produtos.head(10)

,País,produto,valor_usd,litros,preco_medio
0,Afeganistão,Vinho de Mesa,46,11,4.181818
1,Africa do Sul,Uvas Frescas,362,97,3.731959
2,Alemanha,Espumante,316821,64850,4.885443
3,"Alemanha, República Democrática",Uvas Frescas,54085291,25946493,2.084493
4,"Alemanha, República Democrática",Vinho de Mesa,1688896,405963,4.160221
5,"Alemanha, República Democrática da",Suco de Uva,23578,38808,0.607555
6,Angola,Espumante,745652,131937,5.651576
7,Angola,Suco de Uva,239971,187807,1.277753
8,Angola,Uvas Frescas,2670,440,6.068182
9,Angola,Vinho de Mesa,230980,54804,4.214656


In [43]:
top10_preco = preco_por_litro_produtos.groupby('produto').apply(
    lambda x: x.nlargest(10, 'preco_medio')
).reset_index(drop=True)

/tmp/ipykernel_15784/2398112208.py:1: DeprecationWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.



In [44]:
top10_preco.head(10)

,País,produto,valor_usd,litros,preco_medio
0,Catar,Espumante,11117,124,89.653226
1,Bangladesh,Espumante,28,1,28.000000
2,África do Sul,Espumante,1448,82,17.658537
3,Porto Rico,Espumante,36572,2350,15.562553
4,"Tcheca, República",Espumante,6639,527,12.597723
5,Malta,Espumante,54494,4473,12.182875
6,Quênia,Espumante,1996,181,11.027624
7,Tailândia,Espumante,748,70,10.685714
8,Ilha de Man,Espumante,20,2,10.000000
9,Espanha,Espumante,110338,11995,9.198666


In [45]:
fig = px.bar(
    top10_preco,
    x='preco_medio',
    y='País',
    color='produto',
    facet_col='produto',
    facet_col_wrap=2,
    orientation='h',
    title='Top 10 países por preço médio por litro (US$/L) por produto'
)

fig.update_yaxes(matches=None, showticklabels=True)
fig.update_xaxes(matches=None, showticklabels=True)
fig.show()

In [61]:
df_vinho_ativo = df_ativo[df_ativo['produto'] == 'Vinho de Mesa']
consistencia = df_vinho_ativo.groupby('País')['ano'].nunique().sort_values(ascending=False)
print(consistencia.head(20))

País
Alemanha, República Democrática    15
Hong Kong                          15
Bélgica                            15
China                              15
Estados Unidos                     15
Paraguai                           15
Reino Unido                        15
Japão                              15
Itália                             14
Dinamarca                          14
Emirados Árabes Unidos             14
França                             14
Nova Zelândia                      14
Luxemburgo                         14
Canadá                             14
Bolívia                            14
Austrália                          14
Países Baixos                      14
Suriname                           14
Tcheca, República                  14
Name: ano, dtype: int64


In [66]:
consistencia_top20 = consistencia.head(20).reset_index()
consistencia_top20.columns = ['País', 'anos_ativo']

fig = px.bar(
    consistencia_top20,
    x='anos_ativo',
    y='País',
    orientation='h',
    title='Consistência dos parceiros - Vinho de Mesa',
    labels={'anos_ativo': 'Anos com exportação'}
)

fig.update_layout(yaxis={'categoryorder': 'total ascending'})
fig.show()

In [67]:
inconsistentes = consistencia[(consistencia < 15) & (consistencia >= 5)].reset_index()
inconsistentes.columns = ['País', 'anos_ativo']
print(inconsistentes)

             País  anos_ativo
0        Suriname          14
1         Bolívia          14
2          Canada          14
3   Nova Zelândia          14
4        Portugal          14
..            ...         ...
80          Aruba           5
81       Maldivas           5
82         Quênia           5
83          Congo           5
84       Bulgária           5

[85 rows x 2 columns]


In [68]:
por_pais_vinho = df_vinho_ativo.groupby('País')['valor_usd'].sum().reset_index()
por_pais_vinho.columns = ['País', 'valor_usd']

In [69]:
oportunidades = inconsistentes.merge(por_pais_vinho, on='País')
oportunidades = oportunidades.sort_values('valor_usd', ascending=False)
oportunidades.head(15)

,País,anos_ativo,valor_usd
15,Rússia,11,17421596
21,Haiti,10,5174689
13,Espanha,12,3814065
14,Guiana,11,1005911
11,Finlândia,12,537443
3,Portugal,14,456622
22,Curaçao,9,447913
8,Polônia,13,422322
18,Panamá,10,356647
1,Bolívia,14,341425


In [71]:
fig = px.bar(
    oportunidades.head(15),
    x='País',
    y='valor_usd',
    color='anos_ativo',
    title='Parceiros inconsistentes com alto valor - Vinho de Mesa',
    labels={'valor_usd': 'Valor total (US$)', 'anos_ativo': 'Anos ativo'},
    color_continuous_scale='Blues'
)
fig.update_layout(xaxis_tickangle=-45)
fig.show()